# RoSteALS / SepWater — targeted recovery

**A100 (same GPU as the old run) + high RAM.** Resume the reviewed 100-image author-path experiment. Old files are read-only. DistSeal is not run. The separate ZIP contains readable Python, not Base64.

Local checks include real BCH replay on the previous SepWater records. Full CUDA execution remains to be verified in Colab.

## 1. Settings

Leave paths unchanged if the original run is in MyDrive. Only MAX_HOURS is a runtime budget.

In [ ]:
from pathlib import Path
import hashlib, io, os, shutil, subprocess, sys, zipfile

MAX_HOURS = 8  # Session budget, not a promise of completion. May change on resume.
try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if os.name != "posix":
    raise RuntimeError("Use Colab A100 + high RAM or a Linux NVIDIA server.")
BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path("/workspace/data")
PARENT_ROOT = BASE / "watermarking_author_reproduction_v1"  # READ ONLY
OUTPUT_ROOT = BASE / "watermarking_rosteals_sepwater_recovery_v1"
SCRATCH_ROOT = Path("/content/author_recovery_scratch_v1") if IN_COLAB else Path("/workspace/author-recovery-scratch")
SOURCE_ARCHIVE = Path("RoSteALS_SepWater_RECOVERY_source.zip")
SETTINGS = dict(parent=str(PARENT_ROOT), output=str(OUTPUT_ROOT), scratch=str(SCRATCH_ROOT), max_hours=MAX_HOURS)
print("Fixed cohort inherited from the reviewed run: 100 COCO images, 5 messages, full 100-bit payloads.")
print("RoSteALS 256px; SepWater 224px. No DistSeal jobs. No old results will be deleted.")
print("Results:", OUTPUT_ROOT)


## 2. Drive and source setup

Authorize Drive and upload the new matching source ZIP when asked. The old 3-method ZIP is not the new ZIP.

In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")
from IPython.display import Markdown, display
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(SCRATCH_ROOT).free < 35 * 1024**3:
    raise RuntimeError("Need at least 35 GiB free temporary disk. Do not delete Drive results.")
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True))
EXPECTED_SHA = "42ccd248d1764e78cc4a662626b07dab63778d83c56037bf47f60dbcca11c8dd"
cache = OUTPUT_ROOT / "_release" / SOURCE_ARCHIVE.name
if cache.is_file():
    payload = cache.read_bytes()
elif SOURCE_ARCHIVE.is_file():
    payload = SOURCE_ARCHIVE.read_bytes()
elif IN_COLAB:
    print("Upload RoSteALS_SepWater_RECOVERY_source.zip (the matching new source ZIP).")
    uploaded = files.upload()
    candidates = [value for value in uploaded.values() if hashlib.sha256(value).hexdigest() == EXPECTED_SHA]
    if len(candidates) != 1:
        raise ValueError("Upload the matching recovery source ZIP")
    payload = candidates[0]
else:
    raise FileNotFoundError(SOURCE_ARCHIVE)
if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA:
    raise ValueError("Release checksum mismatch; do not mix notebook versions")
# Check path separation before any persistent writes.
paths = [p.resolve() for p in (PARENT_ROOT, OUTPUT_ROOT, SCRATCH_ROOT)]
for i, left in enumerate(paths):
    for right in paths[i+1:]:
        if left.is_relative_to(right) or right.is_relative_to(left):
            raise ValueError("Parent, recovery and scratch directories must be separate")
if not (PARENT_ROOT / "plan.json").is_file():
    raise FileNotFoundError(f"Original author-night run not found: {PARENT_ROOT}")
PROJECT = SCRATCH_ROOT / ("release-" + EXPECTED_SHA[:16])
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for entry in archive.infolist():
        target = (PROJECT / entry.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()) or target == PROJECT.resolve():
            raise ValueError("Unsafe ZIP path")
        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("ZIP symlink rejected")
        if not entry.is_dir():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(entry))
sys.path.insert(0, str(PROJECT))
from authorrecovery.migration import make_plan
from authornight.storage import atomic
PLAN = make_plan(SETTINGS, EXPECTED_SHA)
atomic(cache, payload)
atomic(OUTPUT_ROOT / "DISTSEAL_RESUME.md", (PROJECT / "DISTSEAL_RESUME.md").read_bytes())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "libmagickwand-dev", "build-essential", "libgl1"], check=True)
print("Ready. Run the next cell; imports are resumable and logged every 25 records.")
print("DistSeal handoff saved:", OUTPUT_ROOT / "DISTSEAL_RESUME.md")


## 3. Run / resume

Import verified results → CPU BCH replay → two isolated GPU preflights → missing RoSteALS COCO cases → remaining SepWater/RoSteALS attacks. Only one GPU worker runs at once. Completed verified cases and attack conditions are reused. Before leaving, check for PREFLIGHT PASSED in persistent logs. A time limit means partial coverage, not model failure. If Colab stops, reconnect with the same A100 and run cells 1–3 again. Do not run two copies against one folder.

In [ ]:
from authorrecovery.runner import run
from authorrecovery.report import bundle
try:
    SUMMARY = run(SETTINGS, EXPECTED_SHA)
finally:
    REPORT_ZIP = bundle(OUTPUT_ROOT)
    print("Partial/final report saved:", REPORT_ZIP)


## 4. Report

Full-payload BER, corrected BCH text recovery, technical checks and incomplete coverage are separate. DistSeal continuation instructions are saved in OUTPUT_ROOT/DISTSEAL_RESUME.md.

In [ ]:
from authorrecovery.report import report, bundle
from IPython.display import Markdown, display, FileLink
SUMMARY = report(OUTPUT_ROOT)
display(Markdown((OUTPUT_ROOT / "_reports/report.md").read_text()))
REPORT_ZIP = bundle(OUTPUT_ROOT)
print("Send this report ZIP:", REPORT_ZIP)
if IN_COLAB:
    files.download(str(REPORT_ZIP))
else:
    display(FileLink(str(REPORT_ZIP)))
